# AlphaZero Gomoku - Training on Google Colab

Notebook huấn luyện agent AlphaZero cho game Caro (Gomoku) 15x15.

**Thời gian ước tính:** 6-12 giờ trên Colab GPU (T4/V100)

## Quy trình:
1. Clone repo & cài đặt dependencies
2. Chạy self-play để tạo dữ liệu
3. Train neural network
4. Đánh giá agent vs MiniMax
5. Tải model về máy để chơi

## 1. Setup

In [ ]:
# Clone repo
!git clone https://github.com/VanLam05/Caro-AI.git
%cd Caro-AI

In [ ]:
# Install dependencies
!pip install torch numpy

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Cấu hình Training

Điều chỉnh các hyperparameter tùy theo thời gian bạn muốn train.

| Cấu hình | Iterations | Games/Iter | Simulations | Thời gian ước tính |
|-----------|-----------|------------|-------------|--------------------|
| Nhanh     | 15        | 15         | 100         | ~3-4 giờ           |
| Chuẩn     | 30        | 25         | 200         | ~8-12 giờ          |
| Kỹ        | 50        | 40         | 400         | ~18-24 giờ         |

In [ ]:
# Training configuration
CONFIG = {
    'num_iterations': 30,       # Số vòng lặp training
    'self_play_games': 25,      # Số game self-play mỗi vòng
    'num_simulations': 200,     # Số simulation MCTS mỗi nước đi
    'batch_size': 256,          # Batch size cho training
    'lr': 0.002,                # Learning rate
    'epochs_per_iter': 5,       # Epochs mỗi vòng training
    'num_res_blocks': 5,        # Số residual blocks trong mạng
    'channels': 128,            # Số channels convolution
    'checkpoint_dir': 'neural_net',
}

## 3. Bắt đầu Training

In [ ]:
import sys
sys.path.insert(0, '.')

from pipeline.train import AlphaZeroTrainer

trainer = AlphaZeroTrainer(
    num_iterations=CONFIG['num_iterations'],
    self_play_games=CONFIG['self_play_games'],
    num_simulations=CONFIG['num_simulations'],
    batch_size=CONFIG['batch_size'],
    lr=CONFIG['lr'],
    epochs_per_iter=CONFIG['epochs_per_iter'],
    num_res_blocks=CONFIG['num_res_blocks'],
    channels=CONFIG['channels'],
    checkpoint_dir=CONFIG['checkpoint_dir'],
)

print("Starting training...")
trainer.train()

## 4. Đánh giá kết quả

In [ ]:
# Load trained model and evaluate
from neural_net.architecture import GomokuNet
from mcts.mcts_alpha_zero import MCTS
from game.board import Board
from models.agentMiniMax import AgentMiniMax
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best model
net = GomokuNet(board_size=15, in_channels=4, num_res_blocks=5, channels=128)
net.to(device)
net.load_checkpoint('neural_net/model_checkpoint.pth', device=device)
net.eval()

mcts = MCTS(net, num_simulations=200)

# Evaluate vs MiniMax at different depths
for depth in [1, 3, 5]:
    wins = 0
    num_eval_games = 20
    
    for g in range(num_eval_games):
        board = Board(rows=15, cols=15, winning_condition=5)
        minimax = AgentMiniMax(board, max_depth=depth)
        rl_player = 1 if g % 2 == 0 else -1
        
        move_count = 0
        while True:
            if board.turn == rl_player:
                action, _ = mcts.get_action(board, temperature=0)
                row, col = action
            else:
                move = minimax.get_move()
                if move is None:
                    break
                row, col = move
            
            board.make_move(row, col)
            move_count += 1
            
            result = board.get_game_ended()
            if result != 0:
                if result != 0.5:
                    if (result == 1 and rl_player == board.originXO) or \
                       (result == -1 and rl_player != board.originXO):
                        wins += 1
                break
            
            if move_count >= 225:
                break
    
    print(f"vs MiniMax depth={depth}: {wins}/{num_eval_games} wins ({wins/num_eval_games:.1%})")

## 5. Tải model về máy

Sau khi train xong, tải file `model_checkpoint.pth` về và đặt vào thư mục `neural_net/` trong project.

In [ ]:
# Download model file
from google.colab import files
files.download('neural_net/model_checkpoint.pth')

In [ ]:
# Alternatively, save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('neural_net/model_checkpoint.pth', '/content/drive/MyDrive/model_checkpoint.pth')
print("Model saved to Google Drive!")

## 6. Tiếp tục Training (Resume)

Nếu Colab bị ngắt, bạn có thể tiếp tục train từ checkpoint đã lưu.

In [ ]:
# Resume training from checkpoint
import os

# If you saved to Google Drive, copy back first:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/model_checkpoint.pth', 'neural_net/model_checkpoint.pth')

resume_trainer = AlphaZeroTrainer(
    num_iterations=20,  # Additional iterations
    self_play_games=25,
    num_simulations=200,
    checkpoint_dir='neural_net',
)

checkpoint_path = 'neural_net/model_checkpoint.pth'
if os.path.exists(checkpoint_path):
    resume_trainer.net.load_checkpoint(checkpoint_path, device=resume_trainer.device)
    print(f"Resumed from {checkpoint_path}")

resume_trainer.train()